In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers


In [5]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Paths to your .npz files
TRAIN_PATH = "processed_data/train.npz"
VAL_PATH   = "processed_data/val.npz"
TEST_PATH  = "processed_data/test.npz"

def load_split(npz_path):
    """
    Loads a split from an .npz file and returns:
      X    : np.ndarray, shape = (n_samples, 30, 126)
      mask : np.ndarray, shape = (n_samples, 30)
      y    : list of string labels, length = n_samples
    """
    data = np.load(npz_path, allow_pickle=True)
    X    = data["X"]          # shape: (n_samples, 30, 126)
    mask = data["mask"]       # shape: (n_samples, 30)
    y    = data["y"].tolist() # list of strings
    return X, mask, y

# 1) Load raw arrays, masks, + string labels
X_train, mask_train, y_train_str = load_split(TRAIN_PATH)
X_val,   mask_val,   y_val_str   = load_split(VAL_PATH)
X_test,  mask_test,  y_test_str  = load_split(TEST_PATH)

print("Raw shapes:")
print(f"  X_train:    {X_train.shape},   # samples = {len(y_train_str)}")
print(f"  mask_train: {mask_train.shape}  (# frames per sample = {mask_train.shape[1]})")
print(f"  X_val:      {X_val.shape},     # samples = {len(y_val_str)}")
print(f"  mask_val:   {mask_val.shape}    (# frames per sample = {mask_val.shape[1]})")
print(f"  X_test:     {X_test.shape},    # samples = {len(y_test_str)}")
print(f"  mask_test:  {mask_test.shape}   (# frames per sample = {mask_test.shape[1]})")

Raw shapes:
  X_train:    (1413, 30, 126),   # samples = 1413
  mask_train: (1413, 30)  (# frames per sample = 30)
  X_val:      (260, 30, 126),     # samples = 260
  mask_val:   (260, 30)    (# frames per sample = 30)
  X_test:     (280, 30, 126),    # samples = 280
  mask_test:  (280, 30)   (# frames per sample = 30)


In [6]:
# 2) Encode string labels → integer indices
le = LabelEncoder()
# Fit on all labels (train+val+test) so that index mapping is consistent:
all_labels = y_train_str + y_val_str + y_test_str
le.fit(all_labels)

y_train = le.transform(y_train_str)  # integer array, shape = (n_train,)
y_val   = le.transform(y_val_str)    # shape = (n_val,)
y_test  = le.transform(y_test_str)   # shape = (n_test,)

num_classes = len(le.classes_)
print(f"Detected {num_classes} distinct labels.")


Detected 93 distinct labels.


In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import models, layers, regularizers, callbacks
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [12]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
T_MAX      = 30     # number of frames per example
D_FEATURE  = 126    # 21 keypoints × 2 hands × 3 coords
NUM_CLASSES = 93    # number of distinct sign labels
BATCH_SIZE  = 32
EPOCHS      = 50
LEARNING_RATE = 1e-3
CLIP_NORM     = 1.0
# ──────────────────────────────────────────────────────────────────────────────


def build_sign_model():
    model = models.Sequential([
        # 1) Masking to ignore zero-padded frames
        layers.Masking(mask_value=0.0, input_shape=(T_MAX, D_FEATURE)),

        # 2) First Bi-LSTM layer (return full sequence)
        layers.Bidirectional(
            layers.LSTM(
                units=128,
                return_sequences=True,
                dropout=0.1,            # very light dropout
                recurrent_dropout=0.0   # no recurrent dropout
                # kernel_regularizer removed
            )
        ),

        # 3) Second Bi-LSTM layer (return last hidden state)
        layers.Bidirectional(
            layers.LSTM(
                units=128,
                return_sequences=False,
                dropout=0.1,            # very light dropout
                recurrent_dropout=0.0   # no recurrent dropout
                # kernel_regularizer removed
            )
        ),

        # 4) Dense head with minimal regularization
        layers.Dense(
            units=128,
            activation="relu"
            # kernel_regularizer removed
        ),
        layers.Dropout(0.1),   # small dropout
        layers.BatchNormalization(),

        # 5) Final softmax
        layers.Dense(units=NUM_CLASSES, activation="softmax")
    ])
    return model



# 1) Build & compile:
model = build_sign_model()
optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5
)

history = model.fit(
    x=X_train,         # (N_train, 30, 126)
    y=y_train,         # (N_train,)
    batch_size=32,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/50


/Users/Wilson/miniconda3/lib/python3.11/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


45/45 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - accuracy: 0.0265 - loss: 4.6384 - val_accuracy: 0.0269 - val_loss: 4.3788 - learning_rate: 0.0010
Epoch 2/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.0442 - loss: 4.2302 - val_accuracy: 0.0269 - val_loss: 4.2543 - learning_rate: 0.0010
Epoch 3/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 111ms/step - accuracy: 0.0478 - loss: 4.1040 - val_accuracy: 0.0846 - val_loss: 4.1029 - learning_rate: 0.0010
Epoch 4/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 109ms/step - accuracy: 0.0606 - loss: 3.9162 - val_accuracy: 0.0538 - val_loss: 4.0968 - learning_rate: 0.0010
Epoch 5/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - accuracy: 0.0886 - loss: 3.8872 - val_accuracy: 0.0923 - val_loss: 3.8590 - learning_rate: 0.0010
Epoch 6/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - accuracy: 0.0841 - loss: 3.8053 - val_accuracy: 0.0692 - val_loss: 3.8540 - learning_rate: 0.0010
Epoch 7/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - accuracy: 0.0800 - loss: 3.7666 - val_accuracy:

In [ ]:
# ── Inference & Metrics ──────────────────────────────────────────────────────

# 3) Evaluate on the held-out test set (this prints loss & accuracy):
test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE)
print(f"\nTest loss: {test_loss:.4f}   |   Test accuracy: {test_acc:.4f}")

# 4) Predict class probabilities on X_test:
y_proba = model.predict(X_test, batch_size=BATCH_SIZE)  # shape: (n_test, NUM_CLASSES)

# Convert to integer‐class predictions:
y_pred = np.argmax(y_proba, axis=1)  # shape: (n_test,)

# 5) If you want a human-readable classification report, decode labels back to strings:
#    Assume you used a LabelEncoder 'le' (fitted on all labels) before training:
#        le = LabelEncoder()
#        le.fit(all_labels)  # where all_labels = y_train_str + y_val_str + y_test_str
#    And then y_train = le.transform(y_train_str), etc.
#    Now invert:
le = LabelEncoder()
# (Re‐fit on all string labels just to recover the mapping here:)
all_str_labels = y_train_str + y_val_str + y_test_str
le.fit(all_str_labels)

y_test_str_decoded = le.inverse_transform(y_test)   # length = n_test
y_pred_str_decoded = le.inverse_transform(y_pred)   # length = n_test

# 6) Print a full classification report:
print("\nClassification Report (per‐class):")
print(
    classification_report(
        y_true=y_test_str_decoded,
        y_pred=y_pred_str_decoded,
        digits=4
    )
)

# 7) (Optional) Show confusion matrix as well:
cm = confusion_matrix(y_test_str_decoded, y_pred_str_decoded, labels=le.classes_)
print("\nConfusion Matrix (rows=true, cols=predicted):")
print(cm)